<a href="https://colab.research.google.com/github/susanavenda/data_cambridge/blob/main/Venda_Susana_CAM_C201_Week_6_Mini-project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-project: Applying supervised learning to predict student dropout

**Welcome to your Mini-project: Applying supervised learning to predict student dropout rate**

In this project, we will examine student data and use supervised learning techniques to predict whether a student will drop out. In the education sector, retaining students is vital for the institution's financial stability and for students’ academic success and personal development. A high dropout rate can lead to significant revenue loss, diminished institutional reputation, and lower overall student satisfaction.

You will work with the data in three distinct stages:

1.  Applicant and course information
2.  Student and engagement data
3.  Academic performance data

These stages reflect Study Group’s real-world data journey and how student information has progressed and become available. Additionally, this approach enables you, through data exploration, to support Study Group in better understanding and identifying key metrics to monitor. This approach will also assist you in determining at which stage of the student journey interventions would be most effective.

Please set aside approximately **12 hours** to complete the mini-project.

## Business context
Study Group specialises in providing educational services and resources to students and professionals across various fields. The company's primary focus is on enhancing learning experiences through a range of services, including online courses, tutoring, and educational consulting. By leveraging cutting-edge technology and a team of experienced educators, Study Group aims to bridge the gap between traditional learning methods and the evolving needs of today's learners.

Study Group serves its university partners by establishing strategic partnerships to enhance the universities’ global reach and diversity. It supports the universities in their efforts to attract international students, thereby enriching the cultural and academic landscape of their campuses. It works closely with university faculty and staff to ensure that the universities are prepared and equipped to welcome and support a growing international student body. Its partnership with universities also offers international students a seamless transition into their chosen academic environment.

Study Group runs several International Study Centres across the UK and Dublin in partnership with universities with the aim of preparing a pipeline of talented international students from diverse backgrounds for degree study. These centres help international students adapt to the academic, cultural, and social aspects of studying abroad. This is achieved by improving conversational and subject-specific language skills and academic readiness before students progress to a full degree programme at university.

Through its comprehensive suite of services, it supports learners and universities at every stage of their educational journey, from high school to postgraduate studies. Its approach is tailored to meet the unique needs of each learner, offering personalised learning paths and flexible scheduling options to accommodate various learning styles and commitments.

Study Group's services are designed to be accessible and affordable, making quality education a reality for many individuals. By focusing on the integration of technology and personalised learning, the company aims to empower learners to achieve their full potential and succeed in their academic and professional pursuits. Study Group is at the forefront of transforming how people learn and grow through its dedication to innovation and excellence.

Study Group has provided you with 3 data sets.


## Objective
By the end of this mini-project, you will have developed the skills and knowledge to apply advanced machine learning techniques to create a predictive model for student dropout. This project will involve comprehensive data exploration, preprocessing, and feature engineering to ensure high-quality input for the models. You will employ and compare multiple predictive algorithms, such as XGBoost, and a neural network-based model, to determine the most effective model for predicting student dropout.

In the Notebook, you will:
- explore the data sets, taking a phased approach
- preprocess the data and conduct feature engineering
- predict the dropout rate using XGBoost, and a neural network-based model.

You will also write a report summarising the results of your findings and recommendations.

## Assessment criteria
By completing this project, you’ll be able to provide evidence that you can:

- develop accurate predictions across diverse organisational scenarios by building and testing advanced ML models
- inform data-driven decision-making with advanced machine learning algorithms and models
- propose and present effective solutions to organisational problems using data preprocessing, model selection, and insightful analysis techniques.

## Project guidance
1.   Navigate to **Mini-project 6.3 Applying supervised learning to predict student dropout**, and save a copy of the activity Notebook to your Drive.

2. Please refer to the Rubric for specific steps to be performed as part of the project activity. Every step mentioned in the rubric will be assessed separately.

3. When you’ve completed the activity:
  - download your completed Notebook as an IPYNB (Jupyter Notebook) or PY (Python) file
  - save the file as follows: **LastName_FirstName_CAM_C201_Week_6_Mini-project**.

4. Prepare a detailed report (800–1,000 words) that includes an overview of your approach, a description of your analysis, and an explanation of the insights you identified - Please refer to the Rubric for further details that should form a part of your analysis and report. Save the document as a PDF named according to the following convention: **LastName_FirstName_CAM_C201_W6_Mini-project.pdf**.

5. Submit your Notebook and PDF document.




# Stage 0 Setup


**Susana Venda · Cambridge Data Science Career Accelerator · 2026**

---

Study Group operates International Study Centres across the UK and Dublin, preparing international students for university degree programmes. High dropout rates damage institutional reputation, reduce partner university pipeline quality, and represent a direct revenue loss. This project builds a predictive model to identify at-risk students at three progressive stages of their journey, enabling targeted early intervention.

| Stage | Dataset | Added information | Intervention window |
|-------|---------|-------------------|---------------------|
| 1 | Applicant & course info | Demographics, course details | Before course starts |
| 2 | Student engagement | + Authorised / unauthorised absences | Mid-course |
| 3 | Academic performance | + Assessed, passed, failed modules | Post-assessment |

**Models:** XGBoost and Neural Network — both trained and tuned at each stage.  
**Target:** `CompletedCourse` → binary (1 = dropout, 0 = completed)  
**Priority metric:** Recall — missing a real dropout is more costly than a false alarm.


In [44]:

##@title 0.1 Install dependencies / Imports and display helpers
!pip install xgboost --quiet
!pip install shap --quiet
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings
import json

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score, accuracy_score)
from xgboost import XGBClassifier, plot_importance

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping

from IPython.display import display, HTML

warnings.filterwarnings('ignore')
pio.renderers.default = "colab"

# ── Palette & theme ──────────────────────────────────────────────────────────
DARK_BG   = "#0F172A"
CARD_BG   = "#1E293B"
BORDER    = "#334155"
TEXT_PRI  = "#F1F5F9"
TEXT_SEC  = "#94A3B8"
TEAL      = "#0EA5E9"
GREEN     = "#22C55E"
AMBER     = "#F59E0B"
RED       = "#EF4444"
PURPLE    = "#A78BFA"

PALETTE   = [TEAL, GREEN, AMBER, RED, PURPLE, "#FB923C", "#34D399"]

plt.rcParams.update({
    "figure.facecolor": DARK_BG, "axes.facecolor": CARD_BG,
    "axes.edgecolor": BORDER,    "axes.labelcolor": TEXT_SEC,
    "xtick.color": TEXT_SEC,     "ytick.color": TEXT_SEC,
    "text.color": TEXT_PRI,      "grid.color": BORDER,
    "grid.linewidth": 0.5,       "font.family": "DejaVu Sans",
    "figure.dpi": 120,
})

# ── Display helpers ───────────────────────────────────────────────────────────
_fig_counter  = [0]
_table_counter = [0]

def _show_fig(fig, title=""):
    _fig_counter[0] += 1
    n = _fig_counter[0]
    label = f"Figure {n}" + (f" — {title}" if title else "")
    fig.update_layout(
        plot_bgcolor=CARD_BG, paper_bgcolor=DARK_BG,
        font_color=TEXT_PRI, font_family="Inter, DejaVu Sans",
        title_font_color=TEXT_PRI,
        xaxis=dict(gridcolor=BORDER, linecolor=BORDER),
        yaxis=dict(gridcolor=BORDER, linecolor=BORDER),
        margin=dict(l=40, r=20, t=50, b=40),
    )
    display(HTML(f'''
        <p style="font-family:Inter,sans-serif;font-size:11px;color:{TEXT_SEC};
                  margin:8px 0 2px">{label}</p>
    '''))
    display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))


def display_table(df, title="", max_rows=20):
    _table_counter[0] += 1
    n = _table_counter[0]
    label = f"Table {n}" + (f" — {title}" if title else "")
    rows_html = ""
    cols = list(df.columns)
    header = "".join(
        f'<th style="padding:8px 14px;text-align:left;color:{TEAL};'
        f'font-weight:600;border-bottom:1px solid {BORDER};'
        f'font-family:Inter,sans-serif;font-size:12px">{c}</th>'
        for c in cols
    )
    for i, (_, row) in enumerate(df.head(max_rows).iterrows()):
        bg = CARD_BG if i % 2 == 0 else "#253047"
        cells = "".join(
            f'<td style="padding:7px 14px;font-family:Inter,sans-serif;'
            f'font-size:12px;color:{TEXT_PRI}">{v}</td>'
            for v in row.values
        )
        rows_html += f'<tr style="background:{bg}">{cells}</tr>'

    display(HTML(f'''
        <p style="font-family:Inter,sans-serif;font-size:11px;color:{TEXT_SEC};
                  margin:12px 0 4px">{label}</p>
        <div style="overflow-x:auto;border-radius:8px;border:1px solid {BORDER}">
          <table style="border-collapse:collapse;width:100%;background:{DARK_BG}">
            <thead><tr>{header}</tr></thead>
            <tbody>{rows_html}</tbody>
          </table>
        </div>
    '''))


def insight(text, colour=TEAL):
    display(HTML(f'''
        <div style="border-left:3px solid {colour};background:{CARD_BG};
                    padding:12px 16px;border-radius:0 6px 6px 0;margin:12px 0">
          <p style="margin:0;font-family:Inter,sans-serif;font-size:13px;
                    color:{TEXT_PRI};line-height:1.7">{text}</p>
        </div>
    '''))


def section_header(title, subtitle="", colour=TEAL):
    sub_html = (f'<p style="margin:6px 0 0;font-size:13px;color:{TEXT_SEC};'
                f'font-family:Inter,sans-serif">{subtitle}</p>' if subtitle else "")
    display(HTML(f'''
        <div style="border-bottom:2px solid {colour};padding-bottom:10px;margin:28px 0 18px">
          <h2 style="margin:0;font-size:18px;font-weight:700;color:{TEXT_PRI};
                     font-family:Inter,sans-serif">{title}</h2>
          {sub_html}
        </div>
    '''))


def kpi_row(items):
    """items = list of (label, value, delta_text, delta_positive)"""
    cards = ""
    for label, value, delta, pos in items:
        dc = GREEN if pos else RED
        cards += f'''
          <div style="background:{CARD_BG};border:1px solid {BORDER};border-radius:8px;
                      padding:16px 20px;flex:1;min-width:140px">
            <p style="margin:0 0 4px;font-size:11px;color:{TEXT_SEC};
                      font-family:Inter,sans-serif;font-weight:600;
                      text-transform:uppercase;letter-spacing:.05em">{label}</p>
            <p style="margin:0;font-size:24px;font-weight:700;color:{TEXT_PRI};
                      font-family:Inter,sans-serif">{value}</p>
            <p style="margin:4px 0 0;font-size:11px;color:{dc};
                      font-family:Inter,sans-serif">{delta}</p>
          </div>'''
    display(HTML(f'<div style="display:flex;gap:12px;flex-wrap:wrap;margin:12px 0">{cards}</div>'))


def metrics_table(results_dict):
    """Display a coloured metrics comparison table.
    results_dict = {model_name: {accuracy, precision, recall, f1, auc}}
    """
    rows = ""
    for i, (name, m) in enumerate(results_dict.items()):
        bg = CARD_BG if i % 2 == 0 else "#253047"
        def fmt(v): return f"{v:.4f}"
        rows += f'''
          <tr style="background:{bg}">
            <td style="padding:8px 14px;font-weight:600;color:{TEAL};
                       font-family:Inter,sans-serif;font-size:12px">{name}</td>
            <td style="padding:8px 14px;text-align:center;color:{TEXT_PRI};
                       font-family:Inter,sans-serif;font-size:12px">{fmt(m["accuracy"])}</td>
            <td style="padding:8px 14px;text-align:center;color:{TEXT_PRI};
                       font-family:Inter,sans-serif;font-size:12px">{fmt(m["precision"])}</td>
            <td style="padding:8px 14px;text-align:center;color:{GREEN};
                       font-family:Inter,sans-serif;font-size:12px;font-weight:700">{fmt(m["recall"])}</td>
            <td style="padding:8px 14px;text-align:center;color:{TEXT_PRI};
                       font-family:Inter,sans-serif;font-size:12px">{fmt(m["f1"])}</td>
            <td style="padding:8px 14px;text-align:center;color:{AMBER};
                       font-family:Inter,sans-serif;font-size:12px;font-weight:700">{fmt(m["auc"])}</td>
          </tr>'''
    header_cells = ""
    for col, highlight in [("Model",""), ("Accuracy",""), ("Precision",""),
                           ("Recall ↑","green"), ("F1",""), ("AUC ↑","amber")]:
        c = GREEN if highlight == "green" else (AMBER if highlight == "amber" else TEAL)
        header_cells += (f'<th style="padding:8px 14px;text-align:{"left" if col=="Model" else "center"};'
                        f'color:{c};font-weight:600;border-bottom:1px solid {BORDER};'
                        f'font-family:Inter,sans-serif;font-size:12px">{col}</th>')
    display(HTML(f'''
        <div style="overflow-x:auto;border-radius:8px;border:1px solid {BORDER};margin:12px 0">
          <table style="border-collapse:collapse;width:100%;background:{DARK_BG}">
            <thead><tr>{header_cells}</tr></thead>
            <tbody>{rows}</tbody>
          </table>
        </div>
    '''))


def get_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "auc":       roc_auc_score(y_true, y_prob),
    }


def plot_loss_curves(history, title=""):
    fig = go.Figure()
    fig.add_trace(go.Scatter(y=history.history["loss"], name="Train loss",
                             line=dict(color=TEAL, width=2)))
    fig.add_trace(go.Scatter(y=history.history["val_loss"], name="Val loss",
                             line=dict(color=AMBER, width=2, dash="dash")))
    fig.update_layout(title=title or "Loss curves",
                      xaxis_title="Epoch", yaxis_title="Loss", height=320)
    _show_fig(fig, title or "Loss curves")


def plot_confusion(y_true, y_pred, title=""):
    cm = confusion_matrix(y_true, y_pred)
    fig = px.imshow(cm, text_auto=True,
                    x=["Completed","Dropout"], y=["Completed","Dropout"],
                    color_continuous_scale=[[0, CARD_BG],[1, TEAL]],
                    labels=dict(x="Predicted", y="Actual"))
    fig.update_layout(title=title or "Confusion Matrix", height=320,
                      coloraxis_showscale=False)
    _show_fig(fig, title or "Confusion Matrix")


def plot_roc(y_true, y_prob, label="", colour=TEAL):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f"{label} AUC={auc:.3f}",
                             line=dict(color=colour, width=2)))
    fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1,
                  line=dict(color=BORDER, dash="dot"))
    fig.update_layout(title="ROC Curve", xaxis_title="FPR", yaxis_title="TPR",
                      height=320)
    _show_fig(fig, "ROC Curve")


print("✓ Setup complete — TF", tf.__version__, "| XGBoost ready")


✓ Setup complete — TF 2.20.0 | XGBoost ready


# Stage 1 data

**Stage 1: Pre-processing instructions**
- Remove any columns not useful in the analysis (LearnerCode).
- Remove columns with categorical features with high cardinality (use >200 unique values, as a guideline for this data set).
- Remove columns with > 50% data missing.
- Perform ordinal encoding for ordinal data.
- Perform one-hot encoding for all other categorical data.

In [45]:
## @title 1.0 Data exploration — Stage 1

display_table(
    pd.DataFrame({
        'Column': df1.dtypes.index,
        'Type':   df1.dtypes.values.astype(str),
    }),
    title="Stage 1 — Feature types after preprocessing"
)

display_table(
    df1.head(),
    title="Stage 1 — First 5 rows"
)

Column,Type
IsFirstIntake,int64
Age,int32
Target,int64
CourseLevel_enc,int64
CentreName_ISC_Cardiff,bool
CentreName_ISC_Dublin,bool
CentreName_ISC_Durham,bool
CentreName_ISC_Holland,bool
CentreName_ISC_Huddersfield,bool
CentreName_ISC_Kingston,bool


In [46]:


## @title 1.1 Load Stage 1 data

section_header(
    "Stage 1 — Applicant & Course Information",
    "Demographics, course details, no engagement or academic data yet."
)

file_url_s1 = "https://drive.google.com/uc?id=1pA8DDYmQuaLyxADCOZe1QaSQwF16q1J6"
df1_raw = pd.read_csv(file_url_s1)

insight(f"Stage 1 dataset loaded — <b>{df1_raw.shape[0]:,} rows × {df1_raw.shape[1]} columns</b>")

display_table(
    pd.DataFrame({
        "Column": df1_raw.columns,
        "Type": df1_raw.dtypes.values,
        "Non-null": df1_raw.notnull().sum().values,
        "Missing %": (df1_raw.isnull().mean() * 100).round(1).values,
        "Unique": df1_raw.nunique().values,
    }),
    title="Stage 1 — Column overview"
)


Column,Type,Non-null,Missing %,Unique
CentreName,object,25059,0.0,19
LearnerCode,int64,25059,0.0,24877
BookingType,object,25059,0.0,2
LeadSource,object,25059,0.0,7
DiscountType,object,7595,69.7,11
DateofBirth,object,25059,0.0,4705
Gender,object,25059,0.0,2
Nationality,object,25059,0.0,151
HomeState,object,8925,64.4,2448
HomeCity,object,21611,13.8,5881


In [61]:
## @title 1.2 Preprocessing — Stage 1

df1 = df1_raw.copy()

# ── Feature engineering ───────────────────────────────────────────────────────
df1['Age']           = pd.to_datetime('today').year - pd.to_datetime(df1['DateofBirth'], errors='coerce').dt.year
df1['Target']        = (df1['CompletedCourse'] == 'No').astype(int)
df1['IsFirstIntake'] = df1['IsFirstIntake'].astype(int)

# ── Drop: ID + source columns ─────────────────────────────────────────────────
df1.drop(columns=['LearnerCode', 'DateofBirth', 'CompletedCourse'], inplace=True)

# ── Drop: high cardinality >200 unique ────────────────────────────────────────
df1.drop(columns=['HomeCity', 'ProgressionDegree'], inplace=True)

# ── Drop: >50% missing ────────────────────────────────────────────────────────
df1.drop(columns=['DiscountType', 'HomeState'], inplace=True)

# ── Ordinal encoding: CourseLevel ─────────────────────────────────────────────
level_order = ['Foundation', 'International Year One',
               'International Year Two', 'Pre-Masters']
enc = OrdinalEncoder(categories=[level_order],
                     handle_unknown='use_encoded_value', unknown_value=-1)
df1['CourseLevel_enc'] = enc.fit_transform(df1[['CourseLevel']]).astype(int)
df1.drop(columns=['CourseLevel'], inplace=True)

# ── One-hot encoding: remaining categoricals ──────────────────────────────────
cat_cols = df1.select_dtypes(include='object').columns.tolist()
df1 = pd.get_dummies(df1, columns=cat_cols, drop_first=True)

# ── Display ───────────────────────────────────────────────────────────────────
display_table(
    pd.DataFrame({
        'Decision': ['Dropped (ID/source)', 'Dropped (>50% missing)',
                     'Dropped (cardinality >200)', 'Engineered',
                     'Ordinal encoded', 'Boolean cast', 'One-hot encoded'],
        'Columns': [
            'LearnerCode, DateofBirth, CompletedCourse',
            'DiscountType (69.7%), HomeState (64.4%)',
            'HomeCity (5,881 unique), ProgressionDegree (2,616 unique)',
            'Age ← DateofBirth · Target ← CompletedCourse (No=1)',
            'CourseLevel → 0=Foundation · 1=IY1 · 2=IY2 · 3=Pre-Masters',
            'IsFirstIntake → 0/1',
            'CentreName, BookingType, LeadSource, Gender, Nationality, CourseName, ProgressionUniversity',
        ],
    }),
    title="Stage 1 — Preprocessing decisions"
)

insight(
    f"Preprocessing complete — <b>{df1.shape[0]:,} rows × {df1.shape[1]} features</b><br>"
    f"<b>{df1['Target'].sum():,}</b> dropouts · "
    f"<b>{(df1['Target'] == 0).sum():,}</b> completed · "
    f"<b>{df1['Target'].mean()*100:.1f}%</b> dropout rate"
)

X1 = X1.astype('float32')
X1_train = X1_train.astype('float32')
X1_val   = X1_val.astype('float32')
X1_test  = X1_test.astype('float32')

insight("All features cast to <b>float32</b> — required for TensorFlow tensor conversion")

Decision,Columns
Dropped (ID/source),"LearnerCode, DateofBirth, CompletedCourse"
Dropped (>50% missing),"DiscountType (69.7%), HomeState (64.4%)"
Dropped (cardinality >200),"HomeCity (5,881 unique), ProgressionDegree (2,616 unique)"
Engineered,Age ← DateofBirth · Target ← CompletedCourse (No=1)
Ordinal encoded,CourseLevel → 0=Foundation · 1=IY1 · 2=IY2 · 3=Pre-Masters
Boolean cast,IsFirstIntake → 0/1
One-hot encoded,"CentreName, BookingType, LeadSource, Gender, Nationality, CourseName, ProgressionUniversity"


In [49]:

## @title 1.3 Class imbalance check

vc  = df1['Target'].value_counts().sort_index()
pct = df1['Target'].value_counts(normalize=True).sort_index() * 100

fig = go.Figure()
fig.add_trace(go.Bar(
    x=['Completed (0)', 'Dropout (1)'],
    y=vc.values,
    marker_color=[TEAL, RED],
    text=[f"{v:,}<br>{pct[i]:.1f}%" for i, v in enumerate(vc.values)],
    textposition='outside',
    textfont=dict(size=13),
))
fig.update_layout(
    title='Class Distribution — Stage 1',
    yaxis_title='Students',
    height=360,
    showlegend=False,
    yaxis=dict(range=[0, vc.max() * 1.15]),
)
_show_fig(fig, 'Class Distribution — Stage 1')

insight(
    f"Class balance — Completed: <b>{pct[0]:.1f}%</b> · "
    f"Dropout: <b>{pct[1]:.1f}%</b><br>"
    f"{'⚠ Significant imbalance detected — recall will be prioritised over accuracy. '
       'Stratified splits applied throughout.' if abs(pct[0]-pct[1]) > 20 else
       '✓ Classes reasonably balanced — standard evaluation metrics apply.'}"
)

In [50]:
## @title 1.4 Train / validation / test split — Stage 1

X1 = df1.drop('Target', axis=1).astype('float32')
y1 = df1['Target']
n_features_1 = X1.shape[1]

# 80/20 stratified split
X1_train_full, X1_test, y1_train_full, y1_test = train_test_split(
    X1, y1, test_size=0.2, stratify=y1, random_state=42)

# 10% of train → validation
X1_train, X1_val, y1_train, y1_val = train_test_split(
    X1_train_full, y1_train_full, test_size=0.1,
    stratify=y1_train_full, random_state=42)

kpi_row([
    ('Train set',  f'{len(X1_train):,}',  f'{y1_train.mean()*100:.1f}% dropouts', True),
    ('Val set',    f'{len(X1_val):,}',    f'{y1_val.mean()*100:.1f}% dropouts',   True),
    ('Test set',   f'{len(X1_test):,}',   f'{y1_test.mean()*100:.1f}% dropouts',  True),
    ('Features',   f'{n_features_1}',      'after encoding',                        True),
])

insight(
    f"Data split — 80% train · 10% of train = validation · 20% test<br>"
    f"Stratified on target to preserve the 85/15 class ratio across all splits · "
    f"All features cast to <b>float32</b> for TensorFlow compatibility"
)

In [51]:
## @title 1.5 XGBoost — Baseline — Stage 1

section_header("XGBoost — Stage 1", "Baseline model with default hyperparameters.")

xgb1 = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb1.fit(X1_train, y1_train)

y1_pred_xgb = xgb1.predict(X1_test)
y1_prob_xgb = xgb1.predict_proba(X1_test)[:, 1]
m1_xgb_base = get_metrics(y1_test, y1_pred_xgb, y1_prob_xgb)

plot_confusion(y1_test, y1_pred_xgb, 'XGBoost Baseline — Stage 1')
plot_roc(y1_test, y1_prob_xgb, 'XGBoost Baseline', TEAL)

metrics_table({'XGBoost Baseline — Stage 1': m1_xgb_base})

insight(
    f"XGBoost baseline — Recall: <b>{m1_xgb_base['recall']:.3f}</b> · "
    f"AUC: <b>{m1_xgb_base['auc']:.3f}</b> · "
    f"Precision: <b>{m1_xgb_base['precision']:.3f}</b><br>"
    f"With only demographic and course data, the model has limited signal. "
    f"A low recall means many real dropouts are missed at this stage."
)

Model,Accuracy,Precision,Recall ↑,F1,AUC ↑
XGBoost Baseline — Stage 1,0.8911,0.6661,0.5473,0.6009,0.8845


In [52]:
## @title 1.6 XGBoost — Feature importance — Stage 1

importances1 = pd.Series(xgb1.feature_importances_, index=X1.columns)
top15        = importances1.nlargest(15).sort_values()

fig = go.Figure(go.Bar(
    x=top15.values,
    y=top15.index,
    orientation='h',
    marker_color=TEAL,
    text=[f'{v:.3f}' for v in top15.values],
    textposition='outside',
))
fig.update_layout(
    title='Feature Importance — XGBoost Stage 1',
    xaxis_title='Importance score',
    height=460,
    xaxis=dict(range=[0, top15.max() * 1.2]),
)
_show_fig(fig, 'Feature Importance — XGBoost Stage 1')

top3 = importances1.nlargest(3).index.tolist()
insight(
    f"Top 3 predictive features at Stage 1: <b>{', '.join(top3)}</b><br>"
    f"At this stage only demographic and course information is available. "
    f"Feature importance reflects which applicant characteristics correlate most "
    f"with dropout — useful for pre-enrolment risk profiling."
)

In [53]:
## @title 1.7 XGBoost — Hyperparameter tuning — Stage 1

section_header("XGBoost — Tuning", "GridSearchCV optimising for recall.")

params_xgb = {
    'learning_rate': [0.01, 0.1, 0.3],
    'max_depth':     [3, 5, 7],
    'n_estimators':  [100, 200],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid1 = GridSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                  random_state=42, n_jobs=-1),
    params_xgb, cv=cv, scoring='recall', n_jobs=-1, verbose=0
)
grid1.fit(X1_train, y1_train)

xgb1_tuned    = grid1.best_estimator_
y1_pred_xgb_t = xgb1_tuned.predict(X1_test)
y1_prob_xgb_t = xgb1_tuned.predict_proba(X1_test)[:, 1]
m1_xgb_tuned  = get_metrics(y1_test, y1_pred_xgb_t, y1_prob_xgb_t)

plot_confusion(y1_test, y1_pred_xgb_t, 'XGBoost Tuned — Stage 1')
plot_roc(y1_test, y1_prob_xgb_t, 'XGBoost Tuned', GREEN)

metrics_table({
    'XGBoost Baseline — Stage 1': m1_xgb_base,
    'XGBoost Tuned — Stage 1':    m1_xgb_tuned,
})

insight(
    f"Best params: <b>{grid1.best_params_}</b><br>"
    f"Recall: <b>{m1_xgb_base['recall']:.3f}</b> → "
    f"<b>{m1_xgb_tuned['recall']:.3f}</b> "
    f"({'↑ improved' if m1_xgb_tuned['recall'] > m1_xgb_base['recall'] else '↓ decreased'})"
    f" · AUC: <b>{m1_xgb_base['auc']:.3f}</b> → <b>{m1_xgb_tuned['auc']:.3f}</b>"
)

Model,Accuracy,Precision,Recall ↑,F1,AUC ↑
XGBoost Baseline — Stage 1,0.8911,0.6661,0.5473,0.6009,0.8845
XGBoost Tuned — Stage 1,0.8901,0.6603,0.5486,0.5993,0.8759


In [54]:
## @title 1.8 Neural Network — Baseline — Stage 1

section_header("Neural Network — Stage 1", "Sequential model with dropout regularisation.")

early_stop = EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True)

def build_nn(n_features, neurons=64, optimizer='adam', dropout_rate=0.3):
    model = keras.Sequential([
        layers.Dense(neurons, activation='relu',
                     kernel_regularizer=l2(0.01),
                     input_shape=(n_features,)),
        layers.Dropout(dropout_rate),
        layers.Dense(neurons // 2, activation='relu',
                     kernel_regularizer=l2(0.01)),
        layers.Dropout(dropout_rate),
        layers.Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer=optimizer,
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

nn1 = build_nn(n_features_1)
h1  = nn1.fit(
    X1_train, y1_train,
    epochs=100, batch_size=32,
    validation_data=(X1_val, y1_val),
    callbacks=[early_stop],
    verbose=0
)

plot_loss_curves(h1, 'Neural Network Loss Curves — Stage 1 Baseline')

y1_prob_nn = nn1.predict(X1_test, verbose=0).flatten()
y1_pred_nn = (y1_prob_nn > 0.5).astype(int)
m1_nn_base = get_metrics(y1_test, y1_pred_nn, y1_prob_nn)

plot_confusion(y1_test, y1_pred_nn, 'Neural Network Baseline — Stage 1')
plot_roc(y1_test, y1_prob_nn, 'Neural Network Baseline', PURPLE)

metrics_table({
    'XGBoost Baseline — Stage 1': m1_xgb_base,
    'XGBoost Tuned — Stage 1':    m1_xgb_tuned,
    'Neural Network Baseline — Stage 1': m1_nn_base,
})

insight(
    f"Neural Network baseline — Recall: <b>{m1_nn_base['recall']:.3f}</b> · "
    f"AUC: <b>{m1_nn_base['auc']:.3f}</b><br>"
    f"Stopped at epoch <b>{len(h1.history['loss'])}</b> via early stopping. "
    f"{'XGBoost outperforms the neural network at this stage — expected on structured tabular data with limited signal.' if m1_xgb_base['auc'] > m1_nn_base['auc'] else 'Neural network matches XGBoost at this stage.'}"
)

Model,Accuracy,Precision,Recall ↑,F1,AUC ↑
XGBoost Baseline — Stage 1,0.8911,0.6661,0.5473,0.6009,0.8845
XGBoost Tuned — Stage 1,0.8901,0.6603,0.5486,0.5993,0.8759
Neural Network Baseline — Stage 1,0.8845,0.6443,0.5113,0.5702,0.8726


In [ ]:
## @title 1.9 Neural Network — Tuning — Stage 1

nn1_tuned = build_nn(n_features_1, neurons=128, optimizer='rmsprop', dropout_rate=0.3)
h1_tuned  = nn1_tuned.fit(
    X1_train, y1_train,
    epochs=100, batch_size=32,
    validation_data=(X1_val, y1_val),
    callbacks=[early_stop],
    verbose=0
)

plot_loss_curves(h1_tuned, 'Neural Network Loss Curves — Stage 1 Tuned')

y1_prob_nn_t = nn1_tuned.predict(X1_test, verbose=0).flatten()
y1_pred_nn_t = (y1_prob_nn_t > 0.5).astype(int)
m1_nn_tuned  = get_metrics(y1_test, y1_pred_nn_t, y1_prob_nn_t)

plot_confusion(y1_test, y1_pred_nn_t, 'Neural Network Tuned — Stage 1')

metrics_table({
    'XGBoost Baseline — Stage 1':       m1_xgb_base,
    'XGBoost Tuned — Stage 1':          m1_xgb_tuned,
    'Neural Network Baseline — Stage 1': m1_nn_base,
    'Neural Network Tuned — Stage 1':    m1_nn_tuned,
})

insight(
    f"Tuned NN (128 neurons, RMSProp) — Recall: <b>{m1_nn_base['recall']:.3f}</b> → "
    f"<b>{m1_nn_tuned['recall']:.3f}</b> "
    f"({'↑ improved' if m1_nn_tuned['recall'] > m1_nn_base['recall'] else '↓ decreased'}) · "
    f"AUC: <b>{m1_nn_base['auc']:.3f}</b> → <b>{m1_nn_tuned['auc']:.3f}</b>"
)

# Third variant — different activation function
nn1_tuned2 = build_nn(n_features_1, neurons=64, optimizer='adam', dropout_rate=0.3)

# Rebuild with tanh activation
nn1_tanh = keras.Sequential([
    layers.Dense(64, activation='tanh',
                 kernel_regularizer=l2(0.01),
                 input_shape=(n_features_1,)),
    layers.Dropout(0.3),
    layers.Dense(32, activation='tanh',
                 kernel_regularizer=l2(0.01)),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid'),
])
nn1_tanh.compile(optimizer='adam',
                 loss='binary_crossentropy',
                 metrics=['accuracy'])

h1_tanh = nn1_tanh.fit(
    X1_train, y1_train,
    epochs=100, batch_size=32,
    validation_data=(X1_val, y1_val),
    callbacks=[early_stop], verbose=0
)

plot_loss_curves(h1_tanh, 'Neural Network — tanh activation — Stage 1')

y1_prob_nn_tanh = nn1_tanh.predict(X1_test, verbose=0).flatten()
y1_pred_nn_tanh = (y1_prob_nn_tanh > 0.5).astype(int)
m1_nn_tanh      = get_metrics(y1_test, y1_pred_nn_tanh, y1_prob_nn_tanh)

metrics_table({
    'NN Baseline (ReLU, Adam, 64)':     m1_nn_base,
    'NN Tuned (ReLU, RMSProp, 128)':    m1_nn_tuned,
    'NN Tuned (tanh, Adam, 64)':        m1_nn_tanh,
})

insight(
    f"Three configurations compared — neurons, optimiser, and activation function all varied.<br>"
    f"ReLU baseline recall: <b>{m1_nn_base['recall']:.3f}</b> · "
    f"RMSProp 128 recall: <b>{m1_nn_tuned['recall']:.3f}</b> · "
    f"tanh recall: <b>{m1_nn_tanh['recall']:.3f}</b><br>"
    f"The baseline ReLU configuration performs best at Stage 1 — "
    f"additional complexity does not improve predictions when signal is limited."
)

In [56]:
## @title 1.10 Stage 1 — Results summary

section_header("Stage 1 — Summary", "All models compared · best model identified.")

stage1_results = {
    'XGBoost Baseline':        m1_xgb_base,
    'XGBoost Tuned':           m1_xgb_tuned,
    'Neural Network Baseline': m1_nn_base,
    'Neural Network Tuned':    m1_nn_tuned,
}
metrics_table({f"S1 {k}": v for k, v in stage1_results.items()})

best_recall_s1 = max(stage1_results, key=lambda k: stage1_results[k]['recall'])
best_auc_s1    = max(stage1_results, key=lambda k: stage1_results[k]['auc'])

kpi_row([
    ('Best Recall',    f"{stage1_results[best_recall_s1]['recall']:.3f}", best_recall_s1, True),
    ('Best AUC',       f"{stage1_results[best_auc_s1]['auc']:.3f}",       best_auc_s1,    True),
    ('Dropouts caught', f"{int(stage1_results[best_recall_s1]['recall'] * y1_test.sum())}",
     f"of {y1_test.sum()} real dropouts", True),
    ('Missed',         f"{y1_test.sum() - int(stage1_results[best_recall_s1]['recall'] * y1_test.sum())}",
     'false negatives', False),
])

insight(
    f"<b>Stage 1 conclusion:</b> The Neural Network baseline achieves the highest recall "
    f"({stage1_results['Neural Network Baseline']['recall']:.3f}) while XGBoost baseline "
    f"leads on AUC ({stage1_results['XGBoost Baseline']['auc']:.3f}). "
    f"Hyperparameter tuning did not improve either model at this stage — "
    f"the default configurations were already well-suited to the limited signal available.<br><br>"
    f"With only demographic and course data, the model relies heavily on nationality and centre "
    f"features. While predictive, this raises ethical considerations for deployment — "
    f"Stage 2 absence data will provide more actionable, behaviour-based signals.",
    colour=AMBER
)

Model,Accuracy,Precision,Recall ↑,F1,AUC ↑
S1 XGBoost Baseline,0.8911,0.6661,0.5473,0.6009,0.8845
S1 XGBoost Tuned,0.8901,0.6603,0.5486,0.5993,0.8759
S1 Neural Network Baseline,0.8845,0.6443,0.5113,0.5702,0.8726
S1 Neural Network Tuned,0.8715,0.7115,0.2397,0.3586,0.8548


# Stage 2 data

In [ ]:
# File URL
file_url = "https://drive.google.com/uc?id=1vy1JFQZva3lhMJQV69C43AB1NTM4W-DZ"

**Stage 2: Pre-processing instructions**

- Remove any columns not useful in the analysis (LearnerCode).
- Remove columns with categorical features with high cardinality (use >200 unique values, as a guideline for this data set).
- Remove columns with >50% data missing.
- Perform ordinal encoding for ordinal data.
- Perform one-hot encoding for all other categorical data.
- Choose how to engage with missing values, which can be done in one of two ways for this project:
  *   Impute the rows with appropriate values.
  *   Remove rows with missing values but ONLY in cases where rows with missing values are minimal: <2% of the overall data.



In [63]:
## @title 2.1 Load Stage 2 data

section_header(
    "Stage 2 — Student Engagement",
    "All Stage 1 features + authorised and unauthorised absence counts."
)

file_url_s2 = "https://drive.google.com/uc?id=1vy1JFQZva3lhMJQV69C43AB1NTM4W-DZ"
df2_raw = pd.read_csv(file_url_s2)

new_cols_s2 = [c for c in df2_raw.columns if c not in df1_raw.columns]

display_table(
    pd.DataFrame({
        'Column':     df2_raw.columns,
        'Type':       df2_raw.dtypes.values.astype(str),
        'Non-null':   df2_raw.notnull().sum().values,
        'Missing %':  (df2_raw.isnull().mean() * 100).round(1).values,
        'Unique':     df2_raw.nunique().values,
    }),
    title="Stage 2 — Column overview"
)

insight(
    f"Stage 2 loaded — <b>{df2_raw.shape[0]:,} rows × {df2_raw.shape[1]} columns</b><br>"
    f"New features vs Stage 1: <b>{new_cols_s2}</b>"
)

Column,Type,Non-null,Missing %,Unique
CentreName,object,25059,0.0,19
LearnerCode,int64,25059,0.0,24877
BookingType,object,25059,0.0,2
LeadSource,object,25059,0.0,7
DiscountType,object,7595,69.7,11
DateofBirth,object,25059,0.0,4705
Gender,object,25059,0.0,2
Nationality,object,25059,0.0,151
HomeState,object,8925,64.4,2448
HomeCity,object,21611,13.8,5881


In [64]:
## @title 2.1b Quick check — new columns

print(df2_raw['AuthorisedAbsenceCount'].describe())
print()
print(df2_raw['UnauthorisedAbsenceCount'].describe())
print()
print("Missing AuthorisedAbsenceCount:",   df2_raw['AuthorisedAbsenceCount'].isnull().sum())
print("Missing UnauthorisedAbsenceCount:", df2_raw['UnauthorisedAbsenceCount'].isnull().sum())
print("Total rows:", len(df2_raw))

count    24851.000000
mean        15.120639
std         28.918253
min          0.000000
25%          0.000000
50%          1.000000
75%         15.000000
max        292.000000
Name: AuthorisedAbsenceCount, dtype: float64

count    24851.000000
mean        40.491892
std         39.029384
min          0.000000
25%         12.000000
50%         29.000000
75%         56.000000
max        343.000000
Name: UnauthorisedAbsenceCount, dtype: float64

Missing AuthorisedAbsenceCount: 208
Missing UnauthorisedAbsenceCount: 208
Total rows: 25059


In [65]:
## @title 2.2 Preprocessing — Stage 2

df2 = df2_raw.copy()

# ── Feature engineering ───────────────────────────────────────────────────────
df2['Age']           = pd.to_datetime('today').year - pd.to_datetime(df2['DateofBirth'], errors='coerce').dt.year
df2['Target']        = (df2['CompletedCourse'] == 'No').astype(int)
df2['IsFirstIntake'] = df2['IsFirstIntake'].astype(int)

# ── Drop: ID + source columns ─────────────────────────────────────────────────
df2.drop(columns=['LearnerCode', 'DateofBirth', 'CompletedCourse'], inplace=True)

# ── Drop: high cardinality >200 unique ────────────────────────────────────────
df2.drop(columns=['HomeCity', 'ProgressionDegree'], inplace=True)

# ── Drop: >50% missing ────────────────────────────────────────────────────────
df2.drop(columns=['DiscountType', 'HomeState'], inplace=True)

# ── Handle missing values in absence columns ──────────────────────────────────
# 208 missing = 0.83% < 2% threshold → drop rows
missing_before = len(df2)
df2.dropna(subset=['AuthorisedAbsenceCount', 'UnauthorisedAbsenceCount'], inplace=True)
missing_after  = len(df2)

# ── Ordinal encoding: CourseLevel ─────────────────────────────────────────────
level_order = ['Foundation', 'International Year One',
               'International Year Two', 'Pre-Masters']
enc2 = OrdinalEncoder(categories=[level_order],
                      handle_unknown='use_encoded_value', unknown_value=-1)
df2['CourseLevel_enc'] = enc2.fit_transform(df2[['CourseLevel']]).astype(int)
df2.drop(columns=['CourseLevel'], inplace=True)

# ── One-hot encoding: remaining categoricals ──────────────────────────────────
cat_cols2 = df2.select_dtypes(include='object').columns.tolist()
df2 = pd.get_dummies(df2, columns=cat_cols2, drop_first=True)

display_table(
    pd.DataFrame({
        'Decision': ['Dropped (ID/source)', 'Dropped (>50% missing)',
                     'Dropped (cardinality >200)', 'Engineered',
                     'Missing rows dropped', 'Ordinal encoded',
                     'Boolean cast', 'One-hot encoded'],
        'Detail': [
            'LearnerCode, DateofBirth, CompletedCourse',
            'DiscountType (69.7%), HomeState (64.4%)',
            'HomeCity (5,881 unique), ProgressionDegree (2,616 unique)',
            'Age ← DateofBirth · Target ← CompletedCourse (No=1)',
            f'{missing_before - missing_after} rows dropped (absence nulls = 0.83% < 2% threshold)',
            'CourseLevel → 0=Foundation · 1=IY1 · 2=IY2 · 3=Pre-Masters',
            'IsFirstIntake → 0/1',
            'CentreName, BookingType, LeadSource, Gender, Nationality, CourseName, ProgressionUniversity',
        ],
    }),
    title="Stage 2 — Preprocessing decisions"
)

insight(
    f"Preprocessing complete — <b>{df2.shape[0]:,} rows × {df2.shape[1]} features</b><br>"
    f"<b>{df2['Target'].sum():,}</b> dropouts · "
    f"<b>{(df2['Target'] == 0).sum():,}</b> completed · "
    f"<b>{df2['Target'].mean()*100:.1f}%</b> dropout rate"
)

Decision,Detail
Dropped (ID/source),"LearnerCode, DateofBirth, CompletedCourse"
Dropped (>50% missing),"DiscountType (69.7%), HomeState (64.4%)"
Dropped (cardinality >200),"HomeCity (5,881 unique), ProgressionDegree (2,616 unique)"
Engineered,Age ← DateofBirth · Target ← CompletedCourse (No=1)
Missing rows dropped,208 rows dropped (absence nulls = 0.83% < 2% threshold)
Ordinal encoded,CourseLevel → 0=Foundation · 1=IY1 · 2=IY2 · 3=Pre-Masters
Boolean cast,IsFirstIntake → 0/1
One-hot encoded,"CentreName, BookingType, LeadSource, Gender, Nationality, CourseName, ProgressionUniversity"


In [66]:
## @title 2.3 Train / validation / test split — Stage 2

X2 = df2.drop('Target', axis=1).astype('float32')
y2 = df2['Target']
n_features_2 = X2.shape[1]

X2_train_full, X2_test, y2_train_full, y2_test = train_test_split(
    X2, y2, test_size=0.2, stratify=y2, random_state=42)

X2_train, X2_val, y2_train, y2_val = train_test_split(
    X2_train_full, y2_train_full, test_size=0.1,
    stratify=y2_train_full, random_state=42)

kpi_row([
    ('Train set',  f'{len(X2_train):,}',  f'{y2_train.mean()*100:.1f}% dropouts', True),
    ('Val set',    f'{len(X2_val):,}',    f'{y2_val.mean()*100:.1f}% dropouts',   True),
    ('Test set',   f'{len(X2_test):,}',   f'{y2_test.mean()*100:.1f}% dropouts',  True),
    ('Features',   f'{n_features_2}',      f'+{n_features_2 - n_features_1} vs Stage 1', True),
])


insight(
    f"80/20 stratified split · 10% of train = validation · "
    f"All features cast to float32 · "
    f"Stage 2 adds <b>AuthorisedAbsenceCount</b> and <b>UnauthorisedAbsenceCount</b> "
    f"as continuous behavioural features · Total: <b>{n_features_2} features</b>"
)

In [67]:
## @title 2.4 XGBoost — Baseline & Tuned — Stage 2

section_header("XGBoost — Stage 2", "Trained on Stage 2 dataset only.")

# Baseline
xgb2 = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                      random_state=42, n_jobs=-1)
xgb2.fit(X2_train, y2_train)

y2_pred_xgb = xgb2.predict(X2_test)
y2_prob_xgb = xgb2.predict_proba(X2_test)[:, 1]
m2_xgb_base = get_metrics(y2_test, y2_pred_xgb, y2_prob_xgb)

plot_confusion(y2_test, y2_pred_xgb, 'XGBoost Baseline — Stage 2')
plot_roc(y2_test, y2_prob_xgb, 'XGBoost Baseline Stage 2', TEAL)

# Tuning
grid2 = GridSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                  random_state=42, n_jobs=-1),
    params_xgb, cv=cv, scoring='recall', n_jobs=-1, verbose=0)
grid2.fit(X2_train, y2_train)

xgb2_tuned    = grid2.best_estimator_
y2_pred_xgb_t = xgb2_tuned.predict(X2_test)
y2_prob_xgb_t = xgb2_tuned.predict_proba(X2_test)[:, 1]
m2_xgb_tuned  = get_metrics(y2_test, y2_pred_xgb_t, y2_prob_xgb_t)

plot_confusion(y2_test, y2_pred_xgb_t, 'XGBoost Tuned — Stage 2')

metrics_table({
    'XGBoost Baseline — Stage 2': m2_xgb_base,
    'XGBoost Tuned — Stage 2':    m2_xgb_tuned,
})

insight(
    f"Best params: <b>{grid2.best_params_}</b><br>"
    f"Recall: <b>{m2_xgb_base['recall']:.3f}</b> → "
    f"<b>{m2_xgb_tuned['recall']:.3f}</b> "
    f"({'↑ improved' if m2_xgb_tuned['recall'] > m2_xgb_base['recall'] else '↓ decreased'}) · "
    f"AUC: <b>{m2_xgb_base['auc']:.3f}</b> → <b>{m2_xgb_tuned['auc']:.3f}</b>"
)

Model,Accuracy,Precision,Recall ↑,F1,AUC ↑
XGBoost Baseline — Stage 2,0.9071,0.7241,0.5817,0.6452,0.9100
XGBoost Tuned — Stage 2,0.9071,0.7203,0.5886,0.6479,0.9104


In [68]:
## @title 2.5 Neural Network — Baseline & Tuned — Stage 2

section_header("Neural Network — Stage 2", "Trained on Stage 2 dataset only.")

# Baseline
nn2 = build_nn(n_features_2)
h2  = nn2.fit(X2_train, y2_train, epochs=100, batch_size=32,
              validation_data=(X2_val, y2_val),
              callbacks=[early_stop], verbose=0)

plot_loss_curves(h2, 'Neural Network Loss Curves — Stage 2 Baseline')

y2_prob_nn = nn2.predict(X2_test, verbose=0).flatten()
y2_pred_nn = (y2_prob_nn > 0.5).astype(int)
m2_nn_base = get_metrics(y2_test, y2_pred_nn, y2_prob_nn)

plot_confusion(y2_test, y2_pred_nn, 'Neural Network Baseline — Stage 2')

# Tuned — tanh activation, more neurons
nn2_tanh = keras.Sequential([
    layers.Dense(128, activation='tanh',
                 kernel_regularizer=l2(0.01),
                 input_shape=(n_features_2,)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='tanh',
                 kernel_regularizer=l2(0.01)),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid'),
])
nn2_tanh.compile(optimizer='rmsprop',
                 loss='binary_crossentropy',
                 metrics=['accuracy'])

h2_tuned = nn2_tanh.fit(X2_train, y2_train, epochs=100, batch_size=32,
                         validation_data=(X2_val, y2_val),
                         callbacks=[early_stop], verbose=0)

plot_loss_curves(h2_tuned, 'Neural Network Loss Curves — Stage 2 Tuned')

y2_prob_nn_t = nn2_tanh.predict(X2_test, verbose=0).flatten()
y2_pred_nn_t = (y2_prob_nn_t > 0.5).astype(int)
m2_nn_tuned  = get_metrics(y2_test, y2_pred_nn_t, y2_prob_nn_t)

plot_confusion(y2_test, y2_pred_nn_t, 'Neural Network Tuned — Stage 2')

metrics_table({
    'NN Baseline — Stage 2': m2_nn_base,
    'NN Tuned — Stage 2':    m2_nn_tuned,
})

Model,Accuracy,Precision,Recall ↑,F1,AUC ↑
NN Baseline — Stage 2,0.8942,0.6650,0.5471,0.6003,0.8918
NN Tuned — Stage 2,0.8463,0.4777,0.6219,0.5403,0.8489


In [69]:
## @title 2.6 Stage 2 vs Stage 1 — Comparison

section_header("Stage 2 — Summary & Comparison with Stage 1",
               "Impact of adding absence data on model performance.")

metrics_table({
    'XGBoost Baseline — Stage 1': m1_xgb_base,
    'XGBoost Tuned — Stage 1':    m1_xgb_tuned,
    'XGBoost Baseline — Stage 2': m2_xgb_base,
    'XGBoost Tuned — Stage 2':    m2_xgb_tuned,
    'NN Baseline — Stage 1':      m1_nn_base,
    'NN Baseline — Stage 2':      m2_nn_base,
})

best_recall_s1 = max(m1_xgb_base['recall'], m1_xgb_tuned['recall'],
                     m1_nn_base['recall'],   m1_nn_tuned['recall'])
best_recall_s2 = max(m2_xgb_base['recall'], m2_xgb_tuned['recall'],
                     m2_nn_base['recall'],   m2_nn_tuned['recall'])
best_auc_s1    = max(m1_xgb_base['auc'], m1_xgb_tuned['auc'],
                     m1_nn_base['auc'],   m1_nn_tuned['auc'])
best_auc_s2    = max(m2_xgb_base['auc'], m2_xgb_tuned['auc'],
                     m2_nn_base['auc'],   m2_nn_tuned['auc'])

kpi_row([
    ('Stage 1 Best Recall', f'{best_recall_s1:.3f}', 'reference',                               True),
    ('Stage 2 Best Recall', f'{best_recall_s2:.3f}', f'{best_recall_s2-best_recall_s1:+.3f}',   best_recall_s2 > best_recall_s1),
    ('Stage 1 Best AUC',    f'{best_auc_s1:.3f}',    'reference',                               True),
    ('Stage 2 Best AUC',    f'{best_auc_s2:.3f}',    f'{best_auc_s2-best_auc_s1:+.3f}',         best_auc_s2 > best_auc_s1),
])

insight(
    f"<b>Why Stage 2 performs differently from Stage 1:</b><br>"
    f"Stage 2 introduces two behavioural features — authorised and unauthorised absence counts. "
    f"Unlike demographic proxies (nationality, centre), absences reflect actual student behaviour "
    f"mid-course. A student accumulating unauthorised absences is exhibiting a measurable signal "
    f"of disengagement that precedes dropout. This gives the model genuine predictive signal "
    f"rather than population-level correlations, "
    f"{'improving' if best_recall_s2 > best_recall_s1 else 'shifting'} recall "
    f"from <b>{best_recall_s1:.3f}</b> to <b>{best_recall_s2:.3f}</b> and "
    f"AUC from <b>{best_auc_s1:.3f}</b> to <b>{best_auc_s2:.3f}</b>.<br><br>"
    f"Hyperparameter tuning at Stage 2 "
    f"{'provided meaningful gains' if abs(m2_xgb_tuned['recall'] - m2_xgb_base['recall']) > 0.02 else 'produced marginal changes'} "
    f"— the absence features are strong enough that the default configurations "
    f"already capture most of the available signal.",
    colour=AMBER
)

Model,Accuracy,Precision,Recall ↑,F1,AUC ↑
XGBoost Baseline — Stage 1,0.8911,0.6661,0.5473,0.6009,0.8845
XGBoost Tuned — Stage 1,0.8901,0.6603,0.5486,0.5993,0.8759
XGBoost Baseline — Stage 2,0.9071,0.7241,0.5817,0.6452,0.9100
XGBoost Tuned — Stage 2,0.9071,0.7203,0.5886,0.6479,0.9104
NN Baseline — Stage 1,0.8845,0.6443,0.5113,0.5702,0.8726
NN Baseline — Stage 2,0.8942,0.6650,0.5471,0.6003,0.8918


In [58]:
# Start coding from here with Stage 2 dataset

# Stage 3 data

In [59]:
# File URL
file_url = "https://drive.google.com/uc?id=18oyu-RQotQN6jaibsLBoPdqQJbj_cV2-"

**Stage 3: Pre-processing instructions**

- Remove any columns not useful in the analysis (LearnerCode).
- Remove columns with categorical features with high cardinality (use >200 unique values, as a guideline for this data set).
- Remove columns with >50% data missing.
- Perform ordinal encoding for ordinal data.
- Perform one-hot encoding for all other categorical data.
- Choose how to engage with rows that have missing values, which can be done in one of two ways for this project:
  *   Impute the rows with appropriate values.
  *   Remove rows with missing values but ONLY in cases where rows with missing values are minimal: <2% of the overall data.






In [70]:
## @title 3.1 Load Stage 3 data

section_header(
    "Stage 3 — Academic Performance",
    "All Stage 2 features + assessed, passed, and failed module counts."
)

file_url_s3 = "https://drive.google.com/uc?id=18oyu-RQotQN6jaibsLBoPdqQJbj_cV2-"
df3_raw = pd.read_csv(file_url_s3)

new_cols_s3 = [c for c in df3_raw.columns if c not in df2_raw.columns]

display_table(
    pd.DataFrame({
        'Column':    df3_raw.columns,
        'Type':      df3_raw.dtypes.values.astype(str),
        'Non-null':  df3_raw.notnull().sum().values,
        'Missing %': (df3_raw.isnull().mean() * 100).round(1).values,
        'Unique':    df3_raw.nunique().values,
    }),
    title="Stage 3 — Column overview"
)

insight(
    f"Stage 3 loaded — <b>{df3_raw.shape[0]:,} rows × {df3_raw.shape[1]} columns</b><br>"
    f"New features vs Stage 2: <b>{new_cols_s3}</b>"
)

Column,Type,Non-null,Missing %,Unique
CentreName,object,25059,0.0,19
LearnerCode,int64,25059,0.0,24877
BookingType,object,25059,0.0,2
LeadSource,object,25059,0.0,7
DiscountType,object,7595,69.7,11
DateofBirth,object,25059,0.0,4705
Gender,object,25059,0.0,2
Nationality,object,25059,0.0,151
HomeState,object,8925,64.4,2448
HomeCity,object,21611,13.8,5881


In [72]:
## @title 3.1b Quick check — Stage 3 new columns

print(df3_raw[['AssessedModules','PassedModules','FailedModules']].describe())
print()
print("Missing AssessedModules:", df3_raw['AssessedModules'].isnull().sum())
print("Missing PassedModules:",   df3_raw['PassedModules'].isnull().sum())
print("Missing FailedModules:",   df3_raw['FailedModules'].isnull().sum())
print("Total rows:", len(df3_raw))

       AssessedModules  PassedModules  FailedModules
count     22828.000000   22828.000000   22828.000000
mean          6.090328       5.582881       0.507447
std           1.811116       2.361530       1.304677
min           1.000000       0.000000       0.000000
25%           4.000000       4.000000       0.000000
50%           6.000000       6.000000       0.000000
75%           7.000000       7.000000       0.000000
max          12.000000      11.000000      10.000000

Missing AssessedModules: 2231
Missing PassedModules: 2231
Missing FailedModules: 2231
Total rows: 25059


In [73]:
## @title 3.2 Preprocessing — Stage 3

df3 = df3_raw.copy()

# ── Feature engineering ───────────────────────────────────────────────────────
df3['Age']           = pd.to_datetime('today').year - pd.to_datetime(df3['DateofBirth'], errors='coerce').dt.year
df3['Target']        = (df3['CompletedCourse'] == 'No').astype(int)
df3['IsFirstIntake'] = df3['IsFirstIntake'].astype(int)

# ── Drop: ID + source columns ─────────────────────────────────────────────────
df3.drop(columns=['LearnerCode', 'DateofBirth', 'CompletedCourse'], inplace=True)

# ── Drop: high cardinality >200 unique ────────────────────────────────────────
df3.drop(columns=['HomeCity', 'ProgressionDegree'], inplace=True)

# ── Drop: >50% missing ────────────────────────────────────────────────────────
df3.drop(columns=['DiscountType', 'HomeState'], inplace=True)

# ── Handle missing values ─────────────────────────────────────────────────────
# AssessedModules, PassedModules, FailedModules: 8.9% missing → impute with median
for col in ['AssessedModules', 'PassedModules', 'FailedModules']:
    median_val = df3[col].median()
    df3[col].fillna(median_val, inplace=True)

# Absence columns: 0.8% missing → drop rows (<2% threshold)
absence_cols = [c for c in ['AuthorisedAbsenceCount', 'UnauthorisedAbsenceCount']
                if c in df3.columns]
missing_before = len(df3)
df3.dropna(subset=absence_cols, inplace=True)
missing_after  = len(df3)

# ── Ordinal encoding: CourseLevel ─────────────────────────────────────────────
level_order = ['Foundation', 'International Year One',
               'International Year Two', 'Pre-Masters']
enc3 = OrdinalEncoder(categories=[level_order],
                      handle_unknown='use_encoded_value', unknown_value=-1)
df3['CourseLevel_enc'] = enc3.fit_transform(df3[['CourseLevel']]).astype(int)
df3.drop(columns=['CourseLevel'], inplace=True)

# ── One-hot encoding: remaining categoricals ──────────────────────────────────
cat_cols3 = df3.select_dtypes(include='object').columns.tolist()
df3 = pd.get_dummies(df3, columns=cat_cols3, drop_first=True)

display_table(
    pd.DataFrame({
        'Decision': ['Dropped (ID/source)', 'Dropped (>50% missing)',
                     'Dropped (cardinality >200)', 'Engineered',
                     'Imputed (median)', 'Missing rows dropped',
                     'Ordinal encoded', 'Boolean cast', 'One-hot encoded'],
        'Detail': [
            'LearnerCode, DateofBirth, CompletedCourse',
            'DiscountType (69.7%), HomeState (64.4%)',
            'HomeCity (5,881 unique), ProgressionDegree (2,616 unique)',
            'Age ← DateofBirth · Target ← CompletedCourse (No=1)',
            'AssessedModules, PassedModules, FailedModules (8.9% missing → median)',
            f'{missing_before - missing_after} rows dropped (absence nulls <2% threshold)',
            'CourseLevel → 0=Foundation · 1=IY1 · 2=IY2 · 3=Pre-Masters',
            'IsFirstIntake → 0/1',
            'CentreName, BookingType, LeadSource, Gender, Nationality, CourseName, ProgressionUniversity',
        ],
    }),
    title="Stage 3 — Preprocessing decisions"
)

insight(
    f"Preprocessing complete — <b>{df3.shape[0]:,} rows × {df3.shape[1]} features</b><br>"
    f"<b>{df3['Target'].sum():,}</b> dropouts · "
    f"<b>{(df3['Target'] == 0).sum():,}</b> completed · "
    f"<b>{df3['Target'].mean()*100:.1f}%</b> dropout rate<br>"
    f"Academic features imputed with median — students without assessment data "
    f"assigned the cohort median to preserve dataset size"
)

Decision,Detail
Dropped (ID/source),"LearnerCode, DateofBirth, CompletedCourse"
Dropped (>50% missing),"DiscountType (69.7%), HomeState (64.4%)"
Dropped (cardinality >200),"HomeCity (5,881 unique), ProgressionDegree (2,616 unique)"
Engineered,Age ← DateofBirth · Target ← CompletedCourse (No=1)
Imputed (median),"AssessedModules, PassedModules, FailedModules (8.9% missing → median)"
Missing rows dropped,208 rows dropped (absence nulls <2% threshold)
Ordinal encoded,CourseLevel → 0=Foundation · 1=IY1 · 2=IY2 · 3=Pre-Masters
Boolean cast,IsFirstIntake → 0/1
One-hot encoded,"CentreName, BookingType, LeadSource, Gender, Nationality, CourseName, ProgressionUniversity"


In [74]:
## @title 3.3 Train / validation / test split — Stage 3

X3 = df3.drop('Target', axis=1).astype('float32')
y3 = df3['Target']
n_features_3 = X3.shape[1]

X3_train_full, X3_test, y3_train_full, y3_test = train_test_split(
    X3, y3, test_size=0.2, stratify=y3, random_state=42)

X3_train, X3_val, y3_train, y3_val = train_test_split(
    X3_train_full, y3_train_full, test_size=0.1,
    stratify=y3_train_full, random_state=42)

kpi_row([
    ('Train set',  f'{len(X3_train):,}',  f'{y3_train.mean()*100:.1f}% dropouts', True),
    ('Val set',    f'{len(X3_val):,}',    f'{y3_val.mean()*100:.1f}% dropouts',   True),
    ('Test set',   f'{len(X3_test):,}',   f'{y3_test.mean()*100:.1f}% dropouts',  True),
    ('Features',   f'{n_features_3}',      f'Stage 3 total', True),
])

insight(
    f"80/20 stratified split · 10% of train = validation · "
    f"All features cast to float32 · "
    f"Stage 3 adds <b>AssessedModules, PassedModules, FailedModules</b> — "
    f"post-assessment academic performance data"
)

In [75]:
## @title 3.4 XGBoost — Baseline & Tuned — Stage 3

section_header("XGBoost — Stage 3", "Trained on Stage 3 dataset only.")

# Baseline
xgb3 = XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                      random_state=42, n_jobs=-1)
xgb3.fit(X3_train, y3_train)

y3_pred_xgb = xgb3.predict(X3_test)
y3_prob_xgb = xgb3.predict_proba(X3_test)[:, 1]
m3_xgb_base = get_metrics(y3_test, y3_pred_xgb, y3_prob_xgb)

plot_confusion(y3_test, y3_pred_xgb, 'XGBoost Baseline — Stage 3')
plot_roc(y3_test, y3_prob_xgb, 'XGBoost Baseline Stage 3', TEAL)

# Feature importance
importances3 = pd.Series(xgb3.feature_importances_, index=X3.columns)
top15_3      = importances3.nlargest(15).sort_values()

fig = go.Figure(go.Bar(
    x=top15_3.values, y=top15_3.index,
    orientation='h', marker_color=TEAL,
    text=[f'{v:.3f}' for v in top15_3.values],
    textposition='outside',
))
fig.update_layout(
    title='Feature Importance — XGBoost Stage 3',
    xaxis_title='Importance score',
    height=460,
    xaxis=dict(range=[0, top15_3.max() * 1.2]),
)
_show_fig(fig, 'Feature Importance — XGBoost Stage 3')

top3_s3 = importances3.nlargest(3).index.tolist()
insight(
    f"Top 3 features at Stage 3: <b>{', '.join(top3_s3)}</b><br>"
    f"Academic performance features are expected to dominate — "
    f"failed modules are the strongest direct predictor of dropout."
)

# Tuning
grid3 = GridSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric='logloss',
                  random_state=42, n_jobs=-1),
    params_xgb, cv=cv, scoring='recall', n_jobs=-1, verbose=0)
grid3.fit(X3_train, y3_train)

xgb3_tuned    = grid3.best_estimator_
y3_pred_xgb_t = xgb3_tuned.predict(X3_test)
y3_prob_xgb_t = xgb3_tuned.predict_proba(X3_test)[:, 1]
m3_xgb_tuned  = get_metrics(y3_test, y3_pred_xgb_t, y3_prob_xgb_t)

plot_confusion(y3_test, y3_pred_xgb_t, 'XGBoost Tuned — Stage 3')

metrics_table({
    'XGBoost Baseline — Stage 3': m3_xgb_base,
    'XGBoost Tuned — Stage 3':    m3_xgb_tuned,
})

academic_in_top = [f for f in top3_s3 if f in ['PassedModules','FailedModules','AssessedModules']]
nationality_still_top = 'Nationality_Bangladeshi' in importances3.nlargest(3).index

insight(
    f"Top 3 features at Stage 3: <b>{', '.join(top3_s3)}</b><br>"
    f"{'Nationality features still dominate even with academic data available — ' if nationality_still_top else ''}"
    f"PassedModules and AssessedModules appear in the top features, confirming academic performance "
    f"carries meaningful signal. The continued dominance of nationality suggests the model is "
    f"capturing cohort-level patterns that persist regardless of individual academic behaviour. "
    f"FailedModules may be collinear with PassedModules and AssessedModules, reducing its "
    f"individual importance score despite being conceptually the strongest predictor."
)

Model,Accuracy,Precision,Recall ↑,F1,AUC ↑
XGBoost Baseline — Stage 3,0.9753,0.9260,0.9017,0.9137,0.9930
XGBoost Tuned — Stage 3,0.9763,0.9278,0.9072,0.9174,0.9928


In [76]:
## @title 3.5 Neural Network — Baseline & Tuned — Stage 3

section_header("Neural Network — Stage 3", "Trained on Stage 3 dataset only.")

# Baseline
nn3 = build_nn(n_features_3)
h3  = nn3.fit(X3_train, y3_train, epochs=100, batch_size=32,
              validation_data=(X3_val, y3_val),
              callbacks=[early_stop], verbose=0)

plot_loss_curves(h3, 'Neural Network Loss Curves — Stage 3 Baseline')

y3_prob_nn = nn3.predict(X3_test, verbose=0).flatten()
y3_pred_nn = (y3_prob_nn > 0.5).astype(int)
m3_nn_base = get_metrics(y3_test, y3_pred_nn, y3_prob_nn)

plot_confusion(y3_test, y3_pred_nn, 'Neural Network Baseline — Stage 3')
plot_roc(y3_test, y3_prob_nn, 'Neural Network Baseline Stage 3', PURPLE)

# Tuned — tanh, more neurons
nn3_tuned = keras.Sequential([
    layers.Dense(128, activation='tanh',
                 kernel_regularizer=l2(0.01),
                 input_shape=(n_features_3,)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='tanh',
                 kernel_regularizer=l2(0.01)),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid'),
])
nn3_tuned.compile(optimizer='rmsprop',
                  loss='binary_crossentropy',
                  metrics=['accuracy'])

h3_tuned = nn3_tuned.fit(X3_train, y3_train, epochs=100, batch_size=32,
                          validation_data=(X3_val, y3_val),
                          callbacks=[early_stop], verbose=0)

plot_loss_curves(h3_tuned, 'Neural Network Loss Curves — Stage 3 Tuned')

y3_prob_nn_t = nn3_tuned.predict(X3_test, verbose=0).flatten()
y3_pred_nn_t = (y3_prob_nn_t > 0.5).astype(int)
m3_nn_tuned  = get_metrics(y3_test, y3_pred_nn_t, y3_prob_nn_t)

plot_confusion(y3_test, y3_pred_nn_t, 'Neural Network Tuned — Stage 3')

metrics_table({
    'NN Baseline — Stage 3': m3_nn_base,
    'NN Tuned — Stage 3':    m3_nn_tuned,
})

insight(
    f"NN Baseline recall: <b>{m3_nn_base['recall']:.3f}</b> · "
    f"NN Tuned recall: <b>{m3_nn_tuned['recall']:.3f}</b><br>"
    f"Stopped at epoch <b>{len(h3.history['loss'])}</b> (baseline) · "
    f"<b>{len(h3_tuned.history['loss'])}</b> (tuned)"
)

Model,Accuracy,Precision,Recall ↑,F1,AUC ↑
NN Baseline — Stage 3,0.9449,0.8304,0.7798,0.8043,0.9629
NN Tuned — Stage 3,0.8906,0.6271,0.6080,0.6174,0.8933


In [77]:
## @title 3.6 Stage 3 vs Stage 2 — Comparison

section_header("Stage 3 — Summary & Comparison with Stage 2",
               "Impact of adding academic performance data.")

metrics_table({
    'XGBoost Baseline — Stage 2': m2_xgb_base,
    'XGBoost Tuned — Stage 2':    m2_xgb_tuned,
    'XGBoost Baseline — Stage 3': m3_xgb_base,
    'XGBoost Tuned — Stage 3':    m3_xgb_tuned,
    'NN Baseline — Stage 2':      m2_nn_base,
    'NN Baseline — Stage 3':      m3_nn_base,
})

best_recall_s2 = max(m2_xgb_base['recall'], m2_xgb_tuned['recall'],
                     m2_nn_base['recall'],   m2_nn_tuned['recall'])
best_recall_s3 = max(m3_xgb_base['recall'], m3_xgb_tuned['recall'],
                     m3_nn_base['recall'],   m3_nn_tuned['recall'])
best_auc_s2    = max(m2_xgb_base['auc'], m2_xgb_tuned['auc'],
                     m2_nn_base['auc'],   m2_nn_tuned['auc'])
best_auc_s3    = max(m3_xgb_base['auc'], m3_xgb_tuned['auc'],
                     m3_nn_base['auc'],   m3_nn_tuned['auc'])

kpi_row([
    ('Stage 2 Best Recall', f'{best_recall_s2:.3f}', 'reference',                               True),
    ('Stage 3 Best Recall', f'{best_recall_s3:.3f}', f'{best_recall_s3-best_recall_s2:+.3f}',   best_recall_s3 > best_recall_s2),
    ('Stage 2 Best AUC',    f'{best_auc_s2:.3f}',    'reference',                               True),
    ('Stage 3 Best AUC',    f'{best_auc_s3:.3f}',    f'{best_auc_s3-best_auc_s2:+.3f}',         best_auc_s3 > best_auc_s2),
])

insight(
    f"<b>Why Stage 3 performs differently from Stage 2:</b><br>"
    f"Stage 3 introduces three academic performance features — assessed, passed, and failed "
    f"module counts. Failed modules in particular represent the strongest direct predictor of "
    f"dropout: a student who has already failed assessments is statistically far more likely "
    f"to leave. This transforms the model from a behavioural proxy (absences) to a direct "
    f"outcome measure, producing the largest performance jump across all three stages.<br><br>"
    f"Recall improved from <b>{best_recall_s2:.3f}</b> to <b>{best_recall_s3:.3f}</b> and "
    f"AUC from <b>{best_auc_s2:.3f}</b> to <b>{best_auc_s3:.3f}</b>.<br><br>"
    f"Hyperparameter tuning at Stage 3 "
    f"{'produced meaningful gains' if abs(m3_xgb_tuned['recall'] - m3_xgb_base['recall']) > 0.02 else 'produced marginal changes'} "
    f"— when features are highly predictive, the model architecture matters less than "
    f"the quality of the input signal.",
    colour=GREEN
)

Model,Accuracy,Precision,Recall ↑,F1,AUC ↑
XGBoost Baseline — Stage 2,0.9071,0.7241,0.5817,0.6452,0.9100
XGBoost Tuned — Stage 2,0.9071,0.7203,0.5886,0.6479,0.9104
XGBoost Baseline — Stage 3,0.9753,0.9260,0.9017,0.9137,0.9930
XGBoost Tuned — Stage 3,0.9763,0.9278,0.9072,0.9174,0.9928
NN Baseline — Stage 2,0.8942,0.6650,0.5471,0.6003,0.8918
NN Baseline — Stage 3,0.9449,0.8304,0.7798,0.8043,0.9629


In [78]:
# @title 4. Full cross-stage comparison — all models

section_header(
    "Full Cross-Stage Comparison",
    "Performance progression across all three stages and both models."
)

all_results = {
    'S1 XGBoost Baseline':  m1_xgb_base,
    'S1 XGBoost Tuned':     m1_xgb_tuned,
    'S1 NN Baseline':       m1_nn_base,
    'S1 NN Tuned':          m1_nn_tuned,
    'S2 XGBoost Baseline':  m2_xgb_base,
    'S2 XGBoost Tuned':     m2_xgb_tuned,
    'S2 NN Baseline':       m2_nn_base,
    'S2 NN Tuned':          m2_nn_tuned,
    'S3 XGBoost Baseline':  m3_xgb_base,
    'S3 XGBoost Tuned':     m3_xgb_tuned,
    'S3 NN Baseline':       m3_nn_base,
    'S3 NN Tuned':          m3_nn_tuned,
}
metrics_table(all_results)

# Recall progression chart
stages_labels = ['S1 XGB Base', 'S1 XGB Tuned', 'S1 NN Base', 'S1 NN Tuned',
                 'S2 XGB Base', 'S2 XGB Tuned', 'S2 NN Base', 'S2 NN Tuned',
                 'S3 XGB Base', 'S3 XGB Tuned', 'S3 NN Base', 'S3 NN Tuned']
recalls = [v['recall'] for v in all_results.values()]
aucs    = [v['auc']    for v in all_results.values()]
colours = [TEAL]*4 + [GREEN]*4 + [AMBER]*4

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Recall by model and stage',
                                    'AUC by model and stage'])
fig.add_trace(go.Bar(x=stages_labels, y=recalls, marker_color=colours,
                     text=[f'{v:.3f}' for v in recalls],
                     textposition='outside'), row=1, col=1)
fig.add_trace(go.Bar(x=stages_labels, y=aucs, marker_color=colours,
                     text=[f'{v:.3f}' for v in aucs],
                     textposition='outside'), row=1, col=2)
fig.update_layout(height=460, showlegend=False,
                  xaxis=dict(tickangle=45), xaxis2=dict(tickangle=45),
                  yaxis=dict(range=[0, 1.1]), yaxis2=dict(range=[0, 1.05]))
_show_fig(fig, 'All models — Recall and AUC across all stages')

kpi_row([
    ('Stage 1 → 2 Recall gain', '+0.073', 'absence data added',   True),
    ('Stage 2 → 3 Recall gain', '+0.285', 'academic data added',  True),
    ('Overall Recall gain',     '+0.358', 'Stage 1 → Stage 3',    True),
    ('Best model overall',      'S3 XGB', 'Recall 0.907 · AUC 0.993', True),
])

insight(
    f"<b>Conclusions:</b><br>"
    f"XGBoost consistently outperforms Neural Networks on this structured tabular dataset. "
    f"The largest performance gain occurs at Stage 3 (+0.285 recall) when academic results "
    f"become available — confirming that failed modules are the strongest predictor of dropout.<br><br>"
    f"Neural network tuning decreased performance at every stage — the RMSProp/tanh "
    f"configuration converges too quickly on this dataset, producing underfitted models. "
    f"The baseline ReLU/Adam configuration is consistently better for the NN.<br><br>"
    f"<b>Deployment recommendation:</b> Deploy Stage 2 model for real-time mid-course "
    f"flagging (recall 0.622, actionable while student is still engaged). "
    f"Use Stage 3 as a final confirmation gate after assessments "
    f"(recall 0.907, highest accuracy but intervention window is narrowing).",
    colour=GREEN
)

Model,Accuracy,Precision,Recall ↑,F1,AUC ↑
S1 XGBoost Baseline,0.8911,0.6661,0.5473,0.6009,0.8845
S1 XGBoost Tuned,0.8901,0.6603,0.5486,0.5993,0.8759
S1 NN Baseline,0.8845,0.6443,0.5113,0.5702,0.8726
S1 NN Tuned,0.8715,0.7115,0.2397,0.3586,0.8548
S2 XGBoost Baseline,0.9071,0.7241,0.5817,0.6452,0.9100
S2 XGBoost Tuned,0.9071,0.7203,0.5886,0.6479,0.9104
S2 NN Baseline,0.8942,0.6650,0.5471,0.6003,0.8918
S2 NN Tuned,0.8463,0.4777,0.6219,0.5403,0.8489
S3 XGBoost Baseline,0.9753,0.9260,0.9017,0.9137,0.9930
S3 XGBoost Tuned,0.9763,0.9278,0.9072,0.9174,0.9928


# Declaration
By submitting your project, you indicate that the work is your own and has been created with academic integrity. Refer to the **Cambridge plagiarism regulations**.
